## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the RxNorm Clinical Terminology Mapper model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *RxNorm Clinical Terminology Mapper* offer in the [Azure Marketplace](https://azuremarketplace.microsoft.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-authenticate-online-endpoint) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## RxNorm Clinical Terminology Mapper

- **Model**: [rxnorm_vdb_resolver](https://nlp.johnsnowlabs.com/2024/09/17/rxnorm_resolver_pipeline_en.html)

- **Model Description**: This pretrained pipeline extracts medical entities from clinical text and maps them to their corresponding RxNorm codes.

In [2]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**:
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }
   ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
   {
       "text": "Single text document"
   }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import requests
import json

# Set display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,

    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [4]:
docs = [
    "She is given folic acid 1 mg daily , levothyroxine 0.1 mg and aspirin 81 mg daily.",
    "The patient was advised to take Omeprazole 20 mg daily before breakfast for acid reflux.",
]


sample_text = """He was seen by the endocrinology service and she was discharged on 50 mg of eltrombopag oral at night, 5 mg amlodipine with meals, and metformin 1000 mg two times a day."""

### JSON

#### Example 1: single text document

In [5]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
pd.DataFrame(response_json["predictions"])

,begin,end,ner_chunk,ner_label,ner_confidence,concept_code,resolution,score,concept_class_id,all_codes,concept_name_detailed,all_resolutions,all_score
0,45,91,she was discharged on 50 mg of eltrombopag oral,DRUG,0.6516222,825426,eltrombopag 50 mg,0.808447,"[Clinical Drug Comp, Branded Drug Comp, Branded Drug, Clinical Drug, Clinical Drug Comp]","[825426, 825428, 825429, 825427, 2675198]","[, , , , ]","[eltrombopag 50 mg, eltrombopag 50 mg [promacta], eltrombopag 50 mg oral tablet [promacta], eltrombopag 50 mg oral tablet, eltrombopag 54 mg]","[0.8084465265274048, 0.8038962483406067, 0.782885730266571, 0.7770530581474304, 0.7717964053153992]"
1,103,117,5 mg amlodipine,DRUG,0.5661333,329528,amlodipine 5 mg,0.987806,"[Clinical Drug Comp, Branded Drug Comp, Clinical Drug Comp, Clinical Drug Comp, Branded Drug Comp]","[329528, 2726066, 329527, 2377370, 2726060]","[, , , , ]","[amlodipine 5 mg, amlodipine 5 mg [sdamlo], amlodipine 2.5 mg, levamlodipine 5 mg, amlodipine 2.5 mg [sdamlo]]","[0.9878055453300476, 0.9508147835731506, 0.9416172504425049, 0.9309701323509216, 0.9130842685699463]"
2,135,151,metformin 1000 mg,DRUG,0.6566333,316255,metformin 1000 mg,1.000000,"[Clinical Drug Comp, Clinical Drug Comp, Branded Drug Comp, Branded Drug Comp, Branded Drug Comp]","[316255, 860995, 573342, 541767, 801707]","[, , , , ]","[metformin 1000 mg, metformin hydrochloride 1000 mg, metformin 1000 mg [glucophage], metformin 1000 mg [fortamet], metformin 1000 mg [glumetza]]","[0.9999998807907104, 0.9353883266448975, 0.9338390231132507, 0.9333749413490295, 0.9202077388763428]"


#### Example 2: array of text documents

In [6]:
input_json_data = {"text": docs}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': [[{'begin': 13,
    'end': 27,
    'ner_chunk': 'folic acid 1 mg',
    'ner_label': 'DRUG',
    'ner_confidence': '0.67852503',
    'concept_code': '315966',
    'resolution': 'folic acid 1 mg',
    'score': 1.000000238418579,
    'concept_class_id': ['Clinical Drug Comp',
     'Clinical Drug Comp',
     'Clinical Drug Comp',
     'Branded Drug Comp',
     'Clinical Drug Comp'],
    'all_codes': ['315966', '1087277', '334757', '542061', '336541'],
    'concept_name_detailed': ['', '', '', '', ''],
    'all_resolutions': ['folic acid 1 mg',
     'folic acid 1.1 mg',
     'folic acid 0.1 mg',
     'folic acid 1 mg [folicet]',
     'folic acid 1 mg/ml'],
    'all_score': [1.000000238418579,
     0.9873265027999878,
     0.9669881463050842,
     0.964435875415802,
     0.9637104868888855]},
   {'begin': 37,
    'end': 56,
    'ner_chunk': 'levothyroxine 0.1 mg',
    'ner_label': 'DRUG',
    'ner_confidence': '0.8457667',
    'concept_code': '892245',
    'resolution': 'levo

### JSON Lines

In [7]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines

In [8]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": [{"begin": 13, "end": 27, "ner_chunk": "folic acid 1 mg", "ner_label": "DRUG", "ner_confidence": "0.67852503", "concept_code": "315966", "resolution": "folic acid 1 mg", "score": 1.000000238418579, "concept_class_id": ["Clinical Drug Comp", "Clinical Drug Comp", "Clinical Drug Comp", "Branded Drug Comp", "Clinical Drug Comp"], "all_codes": ["315966", "1087277", "334757", "542061", "336541"], "concept_name_detailed": ["", "", "", "", ""], "all_resolutions": ["folic acid 1 mg", "folic acid 1.1 mg", "folic acid 0.1 mg", "folic acid 1 mg [folicet]", "folic acid 1 mg/ml"], "all_score": [1.000000238418579, 0.9873265027999878, 0.9669881463050842, 0.964435875415802, 0.9637104868888855]}, {"begin": 37, "end": 56, "ner_chunk": "levothyroxine 0.1 mg", "ner_label": "DRUG", "ner_confidence": "0.8457667", "concept_code": "892245", "resolution": "levothyroxine sodium 0.1 mg", "score": 0.9439408779144287, "concept_class_id": ["Clinical Drug Comp", "Branded Drug Comp", "Branded Drug Com